# T5 文本摘要

这个 Notebook 展示 `T5（Text-to-Text Transfer Transformer）` 做文本摘要的完整流程。

内容包括：
- T5 Encoder-Decoder 架构解读
- 与 BERT（Encoder-Only）和 GPT-2（Decoder-Only）的三角对比
- Text-to-Text 统一框架思路
- 解码策略：greedy / beam search / sampling
- 在 CNN/DailyMail 摘要任务上 fine-tune
- ROUGE 评估结果展示

## 1. 环境准备

```bash
pip install torch transformers datasets rouge-score
```

In [ ]:
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import T5ForConditionalGeneration, T5Tokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from datasets import load_dataset

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
@dataclass
class Config:
    model_name: str = 't5-small'
    max_input_length: int = 512
    # 摘要通常远短于原文
    max_target_length: int = 128
    batch_size: int = 8
    num_workers: int = 2
    lr: float = 3e-4
    epochs: int = 3
    # 控制数据量，加快实验周期
    train_samples: int = 1000
    val_samples: int = 200
    # beam search 候选数量
    num_beams: int = 4

cfg = Config()
cfg

## 2. 加载数据

使用 CNN/DailyMail 数据集，包含新闻文章和对应摘要。

In [ ]:
raw = load_dataset('cnn_dailymail', '3.0.0')
train_raw = raw['train'].select(range(cfg.train_samples))
val_raw   = raw['validation'].select(range(cfg.val_samples))

print(f'训练集：{len(train_raw)} 篇')
print(f'验证集：{len(val_raw)} 篇')
print('\n示例文章（前 200 字）：')
print(train_raw[0]['article'][:200])
print('\n参考摘要：')
print(train_raw[0]['highlights'])

## 3. Tokenizer 与 DataLoader

In [ ]:
tokenizer = T5Tokenizer.from_pretrained(cfg.model_name)


class SummarizationDataset(Dataset):
    def __init__(self, data, tokenizer, cfg):
        self.data = data
        self.tokenizer = tokenizer
        self.cfg = cfg

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        # T5 用前缀区分任务，这是 Text-to-Text 框架的关键设计
        input_text  = 'summarize: ' + item['article']
        target_text = item['highlights']

        enc = self.tokenizer(
            input_text,
            max_length=self.cfg.max_input_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        tgt = self.tokenizer(
            target_text,
            max_length=self.cfg.max_target_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        labels = tgt['input_ids'].squeeze(0)
        # padding token 不参与损失计算
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'labels':         labels,
        }


train_dataset = SummarizationDataset(train_raw, tokenizer, cfg)
val_dataset   = SummarizationDataset(val_raw,   tokenizer, cfg)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True,  num_workers=cfg.num_workers)
val_loader   = DataLoader(val_dataset,   batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

print('DataLoader 构建完成')

## 4. T5 结构解读

### 4.1 三种主流 Transformer 架构对比

| 架构 | 代表模型 | 注意力 | 适合任务 |
|------|---------|--------|----------|
| Encoder-Only | BERT | 双向全注意力 | 分类、NER、问答 |
| Decoder-Only | GPT-2, LLaMA | 单向（自回归） | 文本生成、续写 |
| Encoder-Decoder | T5, BART | Enc 双向 + Dec 单向 + 交叉注意力 | 翻译、摘要、问答 |

### 4.2 Text-to-Text 统一框架

T5 的核心创新：**所有 NLP 任务都转化为 `text -> text`**

| 任务 | 输入前缀 | 示例输入 | 输出 |
|------|----------|----------|------|
| 摘要 | `summarize:` | `summarize: The president...` | `Biden said...` |
| 翻译 | `translate English to French:` | `translate ...: Hello` | `Bonjour` |
| 分类 | `sst2 sentence:` | `sst2 sentence: great movie` | `positive` |

### 4.3 Encoder-Decoder 交叉注意力

Decoder 每步生成时，通过 **Cross-Attention** 查询 Encoder 的输出，决定当前词应该关注原文的哪些部分。

In [ ]:
model = T5ForConditionalGeneration.from_pretrained(cfg.model_name).to(device)
print(model.config)

## 5. 解码策略解读

In [ ]:
# 用同一段文本演示三种解码策略的差异
sample_text = 'summarize: ' + train_raw[0]['article'][:800]
inputs = tokenizer(sample_text, return_tensors='pt', max_length=512, truncation=True).to(device)

model.eval()
with torch.no_grad():
    # Greedy：每步取概率最大的词，快但容易重复
    greedy_ids = model.generate(**inputs, max_new_tokens=80)

    # Beam Search：维护多条候选序列，平衡质量与多样性
    beam_ids = model.generate(**inputs, max_new_tokens=80, num_beams=cfg.num_beams)

    # Top-p Sampling：从累积概率超过 p 的词中随机采样，更有创造性
    sample_ids = model.generate(**inputs, max_new_tokens=80, do_sample=True, top_p=0.92, temperature=0.8)

print('=== Greedy ===' )
print(tokenizer.decode(greedy_ids[0], skip_special_tokens=True))
print('\n=== Beam Search (beams=4) ===')
print(tokenizer.decode(beam_ids[0], skip_special_tokens=True))
print('\n=== Top-p Sampling ===')
print(tokenizer.decode(sample_ids[0], skip_special_tokens=True))

## 6. 参数量统计

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total = count_parameters(model)
print(f'T5-small 可训练参数：{total:,}（约 {total/1e6:.0f}M）')

## 7. 训练函数

In [ ]:
optimizer = AdamW(model.parameters(), lr=cfg.lr, weight_decay=0.01)
total_steps = len(train_loader) * cfg.epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps)


def train_one_epoch(model, dataloader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / len(dataloader)


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_loss += outputs.loss.item()
    return total_loss / len(dataloader)

## 8. 训练主循环

In [ ]:
history = {'train_loss': [], 'val_loss': []}

for epoch in range(cfg.epochs):
    tr_loss  = train_one_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss = evaluate(model, val_loader, device)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    print(f'Epoch {epoch+1}/{cfg.epochs}  train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history['train_loss'], label='train')
ax.plot(history['val_loss'],   label='val')
ax.set_title('T5 Fine-tune 损失曲线')
ax.set_xlabel('Epoch')
ax.legend()
plt.tight_layout()
plt.show()

## 9. 摘要结果展示与 ROUGE 评估

ROUGE（Recall-Oriented Understudy for Gisting Evaluation）是摘要任务常用自动评估指标：
- **ROUGE-1**：基于单词（unigram）的重叠率
- **ROUGE-2**：基于二元组（bigram）的重叠率  
- **ROUGE-L**：最长公共子序列，衡量句子结构相似度

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

@torch.no_grad()
def generate_and_score(model, tokenizer, samples, cfg, device, n=5):
    model.eval()
    r1_scores, r2_scores, rL_scores = [], [], []

    for i in range(min(n, len(samples))):
        article = 'summarize: ' + samples[i]['article']
        reference = samples[i]['highlights']

        inputs = tokenizer(article, return_tensors='pt', max_length=512, truncation=True).to(device)
        ids = model.generate(**inputs, max_new_tokens=cfg.max_target_length, num_beams=cfg.num_beams)
        hypothesis = tokenizer.decode(ids[0], skip_special_tokens=True)

        scores = scorer.score(reference, hypothesis)
        r1_scores.append(scores['rouge1'].fmeasure)
        r2_scores.append(scores['rouge2'].fmeasure)
        rL_scores.append(scores['rougeL'].fmeasure)

        print(f'--- 样本 {i+1} ---')
        print(f'参考：{reference[:120]}...')
        print(f'生成：{hypothesis[:120]}...')
        print(f'ROUGE-1={scores["rouge1"].fmeasure:.3f}  ROUGE-2={scores["rouge2"].fmeasure:.3f}  ROUGE-L={scores["rougeL"].fmeasure:.3f}\n')

    print(f'平均 ROUGE-1={sum(r1_scores)/len(r1_scores):.3f}  '
          f'ROUGE-2={sum(r2_scores)/len(r2_scores):.3f}  '
          f'ROUGE-L={sum(rL_scores)/len(rL_scores):.3f}')


generate_and_score(model, tokenizer, val_raw, cfg, device)